1. Download and Run Open-Source Models
    - Use Hugging Face Transformers to download Llama3/4 or Mistral

In [1]:
import os

from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load Hugging Face token from .env file
load_dotenv()
hf_hub_token = os.getenv("HUGGINGFACE_HUB_TOKEN")

if not hf_hub_token:
    raise ValueError("HUGGINGFACE_HUB_TOKEN not found in environment variables.")

# Model and tokenizer identifiers
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_hub_token)

# Load model with automatic device mapping and precision
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_hub_token,
    device_map="auto",  # Automatically selects GPU if available
    torch_dtype="auto",  # Uses float16 on GPU, float32 on CPU
)

# Define prompt
prompt = "The Eiffel Tower is located in"

# Tokenize input and move to model device
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {key: value.to(model.device) for key, value in inputs.items()}

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=10,
    do_sample=False  # Deterministic output; set to True for sampling
)

# Decode and display result
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

/home/carlo/miniconda3/envs/.conda-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:07<00:00,  1.90s/it]
Some parameters are on the meta device because they were offloaded to the disk and cpu.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


The Eiffel Tower is located in Paris, France, and is one of the most


2. Try vLLM Serving
    - Install vllm
    - Start server

In [ ]:
! pip install vllm --torch-backend=auto
! pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
! python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen1.5-1.8B-Chat \
  --gpu-memory-utilization 0.8 \
  --max-model-len 1024 \
  --dtype float16 \
  --api-key token-local

INFO 09-03 10:55:28 [__init__.py:241] Automatically detected platform cuda.
(APIServer pid=7234) INFO 09-03 10:55:30 [api_server.py:1805] vLLM API server version 0.10.1.1
(APIServer pid=7234) INFO 09-03 10:55:30 [utils.py:326] non-default args: {'model': 'microsoft/phi-3-mini-4k-instruct', 'dtype': 'float16', 'max_model_len': 512, 'gpu_memory_utilization': 0.6}
(APIServer pid=7234) INFO 09-03 10:55:39 [__init__.py:711] Resolved architecture: Phi3ForCausalLM
(APIServer pid=7234) `torch_dtype` is deprecated! Use `dtype` instead!
(APIServer pid=7234) WARNING 09-03 10:55:39 [__init__.py:2819] Casting torch.bfloat16 to torch.float16.
(APIServer pid=7234) INFO 09-03 10:55:39 [__init__.py:1750] Using max model len 512
(APIServer pid=7234) INFO 09-03 10:55:40 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=2048.
(APIServer pid=7234) WARNING 09-03 10:55:40 [cache.py:216] Possibly too large swap space. 4.00 GiB out of the 7.76 GiB total CPU memory is allocated for the s

In [11]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="token-local" # dummy key
)

response = client.chat.completions.create(
    model="Qwen/Qwen1.5-1.8B-Chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What's the capital of Canada?"}
    ]
)

print(response.choices[0].message.content)

The capital of Canada is Ottawa. It is located in eastern Ontario, on the east bank of the Ottawa River, near the provincial capital city of Toronto. The city has been the official capital of Canada since 1867 and was originally established as a trading post for British Columbia fur traders. Today, Ottawa is a bustling metropolis with a population of over 900,000 people, known for its political institutions, museums, landmarks, and cultural attractions such as the National Gallery of Canada and the Rideau Canal. The city is also home to many major corporations, educational institutions, and government agencies, including the Canadian Parliament, which is headquartered there.


3. Compare Results  
   - Use your prompt from Project 1
   - Evaluate how local models stack up vs Hugging Face in accuracy, speed, and cost

In [4]:
import time

import requests

# === Local Qwen model config ===
LOCAL_QWEN_URL = "http://localhost:8000/v1/chat/completions"
LOCAL_QWEN_KEY = "token-local"
LOCAL_QWEN_NAME = "Qwen/Qwen1.5-1.8B-Chat"

# === Ollama model config ===
OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "mistral:7b-instruct"  # Change to any model you pulled

# === Prompts to test ===
prompts = [
    "What is the capital of Canada?",
    "Explain the theory of relativity in simple terms.",
    "Write a haiku about autumn in Ontario."
]

# === Query local Qwen ===
def query_qwen(prompt):
    headers = {"Authorization": f"Bearer {LOCAL_QWEN_KEY}"}
    payload = {
        "model": LOCAL_QWEN_NAME,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    }
    start = time.time()
    r = requests.post(LOCAL_QWEN_URL, headers=headers, json=payload)
    latency = time.time() - start
    try:
        content = r.json()["choices"][0]["message"]["content"]
    except Exception:
        content = "[Error parsing Qwen output]"
    return content.strip(), latency

# === Query Ollama model ===
def query_ollama(prompt):
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        "stream": False
    }
    start = time.time()
    r = requests.post(OLLAMA_URL, json=payload)
    latency = time.time() - start
    try:
        content = r.json()["message"]["content"]
    except Exception:
        content = "[Error parsing Ollama output]"
    return content.strip(), latency

# === Run comparison ===
for prompt in prompts:
    print(f"\nPrompt: {prompt}\n")

    qwen_resp, qwen_time = query_qwen(prompt)
    print(f"Qwen ({qwen_time:.2f}s):\n{qwen_resp}\n")

    ollama_resp, ollama_time = query_ollama(prompt)
    print(f"Ollama {OLLAMA_MODEL} ({ollama_time:.2f}s):\n{ollama_resp}\n")

    print("-" * 60)


Prompt: What is the capital of Canada?

Qwen (4.15s):
The capital city of Canada is Ottawa. It is located in eastern Ontario on the banks of the Ottawa River, just south of its western border with Quebec City. The city has a population of approximately 625,000 people and serves as the federal seat of government for Canada. 

Ottawa is known for its rich history, including the founding of Canada as a Dominion in 1867 and its role in shaping Canadian national identity. The city is home to many important institutions, including the National Museum of Canada, the Supreme Court of Canada, and the Parliament Hill building, which houses the legislative branch of the Canadian government.

In addition to its political significance, Ottawa is also a cultural hub, with numerous museums, galleries, theaters, and other attractions that showcase the country's diverse arts scene. The city is known for its scenic parks, such as Carlsbad Park and Rideau Riverfront, which provide opportunities for outd